# E2E Type Progression

This notebook illustrates the typed interface progression used by `mellea-lrc`:

1. `Path` -> `PreprocessedDocument`
2. `PreprocessedDocument` -> `ExtractedDocument`
3. `ExtractedDocument` -> `ValidatedDocument`
4. `ValidatedDocument` -> `AssessedDocument`

Each step writes `local/snapshots/<doc>/<stage>.json`, reloads through `deserialize_*`, and reads its input from the previous snapshot. Re-run any step after setup as long as upstream snapshots exist.

In [1]:
from pathlib import Path

from IPython.display import display

# Configuration
USE_BOOKMARKED = True  # True: bookmarked file; False: indexed test-data file
TEST_DATA_INDEX = 0
TEST_DATA_DIR = Path("../local/test_data")
BOOKMARKED_PATH = Path("../local/bookmarked/bookmarked.txt")
SNAPSHOT_ROOT = Path("../local/snapshots")
MELLEA_CONCURRENCY = 5

TEST_DATA_FILES = sorted(
    TEST_DATA_DIR.glob("*.txt"),
    key=lambda path: int(path.stem) if path.stem.isdigit() else path.stem,
)
if USE_BOOKMARKED:
    if not BOOKMARKED_PATH.is_file():
        raise FileNotFoundError(f"Missing bookmarked file: {BOOKMARKED_PATH.resolve()}")
    DOC_PATH = BOOKMARKED_PATH
else:
    if not TEST_DATA_FILES:
        raise FileNotFoundError(f"No .txt files found in {TEST_DATA_DIR.resolve()}")
    if not 0 <= TEST_DATA_INDEX < len(TEST_DATA_FILES):
        raise IndexError(
            f"TEST_DATA_INDEX={TEST_DATA_INDEX} is outside 0..{len(TEST_DATA_FILES) - 1}"
        )
    DOC_PATH = TEST_DATA_FILES[TEST_DATA_INDEX]

SNAPSHOT_DIR = SNAPSHOT_ROOT / DOC_PATH.stem

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

display(
    {
        "doc_path": str(DOC_PATH),
        "snapshot_dir": str(SNAPSHOT_DIR),
    }
)

{'doc_path': '../local/bookmarked/bookmarked.txt',
 'snapshot_dir': '../local/snapshots/bookmarked'}

In [2]:
from __future__ import annotations

import json
import os
from pathlib import Path


from mellea_lrc.core.env import load_env_file

load_env_file(Path("../.env"), override=True)

from mellea_lrc.assessment import (
    CitationAssessmentResult,
    MelleaCallContext,
    run_assessment_async,
)
from mellea_lrc.extraction import run_extraction
from mellea_lrc.llm import llm_api_config_from_env
from mellea_lrc.preprocessing import run_preprocessing
from mellea_lrc.serialization import (
    deserialize_assessed_document,
    deserialize_extracted_document,
    deserialize_preprocessed_document,
    deserialize_validated_document,
    serialize_assessed_document,
    serialize_extracted_document,
    serialize_preprocessed_document,
    serialize_validated_document,
)
from mellea_lrc.validation import run_validation



from collections.abc import Callable

SNAPSHOT_STAGES = ("preprocessed", "extraction", "validation", "assessment")


def snapshot_path(stage: str) -> Path:
    if stage not in SNAPSHOT_STAGES:
        msg = f"Unknown snapshot stage: {stage}"
        raise ValueError(msg)
    return SNAPSHOT_DIR / f"{stage}.json"


def write_snapshot(stage: str, payload: object) -> Path:
    path = snapshot_path(stage)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def load_snapshot(stage: str, deserialize: Callable[[dict[str, object]], object]) -> object:
    path = snapshot_path(stage)
    if not path.exists():
        msg = f"Missing snapshot: {path}"
        raise FileNotFoundError(msg)
    payload = json.loads(path.read_text(encoding="utf-8"))
    return deserialize(payload)


def summarize_snapshot(stage: str, path: Path, payload: dict[str, object]) -> dict[str, object]:
    summary = {
        "stage": stage,
        "snapshot": str(path),
        "top_level_keys": sorted(payload.keys()),
    }
    counts = payload.get("counts")
    if isinstance(counts, dict):
        summary["counts"] = counts
    return summary

## Step 1: Preprocess

`run_preprocessing(Path)` produces the first typed boundary object: `PreprocessedDocument`. The snapshot is reloaded with `deserialize_preprocessed_document` before extraction.

In [3]:
preprocessed_raw = run_preprocessing(DOC_PATH)
preprocessed_payload = serialize_preprocessed_document(preprocessed_raw)
preprocessed_path = write_snapshot("preprocessed", preprocessed_payload)
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)

display(type(preprocessed))
display(
    {
        **summarize_snapshot("preprocessed", preprocessed_path, preprocessed_payload),
        "chars": len(preprocessed.text),
        "source_path": preprocessed.source_metadata.path,
        "source_format": preprocessed.source_metadata.format.value,
        "backend": preprocessed.preprocessing_metadata.backend.value,
    }
)

mellea_lrc.preprocessing.types.PreprocessedDocument

{'stage': 'preprocessed',
 'snapshot': '../local/snapshots/bookmarked/preprocessed.json',
 'top_level_keys': ['artifact_type',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text'],
 'chars': 1037,
 'source_path': '../local/bookmarked/bookmarked.txt',
 'source_format': 'text',
 'backend': 'plain_text'}

## Step 2: Extract

`run_extraction(PreprocessedDocument)` advances the object to `ExtractedDocument`. The snapshot is reloaded with `deserialize_extracted_document` before validation.

In [4]:
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)
extraction_raw = run_extraction(preprocessed)
extraction_payload = serialize_extracted_document(extraction_raw)
extraction_path = write_snapshot("extraction", extraction_payload)
extraction = load_snapshot("extraction", deserialize_extracted_document)

display(type(extraction))
display(summarize_snapshot("extraction", extraction_path, extraction_payload))
display(extraction_payload["citations"][:3])

mellea_lrc.extraction.types.ExtractedDocument

{'stage': 'extraction',
 'snapshot': '../local/snapshots/bookmarked/extraction.json',
 'top_level_keys': ['artifact_type',
  'citations',
  'extraction_metadata',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text']}

[{'citation_id': 'cite-0001',
  'span': {'start': 209, 'end': 369},
  'matched_text': '806 F.3d 1288',
  'citation': {'plaintiff': 'United States',
   'defendant': 'Hoffman',
   'volume': '806',
   'reporter': 'F.3d',
   'page': '1288',
   'pin_cite': '1295',
   'extra': None,
   'year': '2015',
   'court': 'ca10',
   'parenthetical': 'distinguishing between prohibited credibility attacks and permissible non-credibility purposes',
   'type': 'FullCaseCitation'},
  'resolves_to': None},
 {'citation_id': 'cite-0002',
  'span': {'start': 725, 'end': 770},
  'matched_text': '2019 WL 1085179',
  'citation': {'plaintiff': None,
   'defendant': None,
   'volume': '2019',
   'reporter': 'WL',
   'page': '1085179',
   'pin_cite': 'at *78',
   'extra': None,
   'year': '2019',
   'court': 'nmd',
   'parenthetical': None,
   'type': 'FullCaseCitation'},
  'resolves_to': None},
 {'citation_id': 'cite-0003',
  'span': {'start': 779, 'end': 837},
  'matched_text': '950 F.3d 754',
  'citation': {'pla

## Step 3: Validate

`run_validation(ExtractedDocument)` advances the object to `ValidatedDocument`. The snapshot is reloaded with `deserialize_validated_document` before assessment.

This may call the configured CourtListener access service.

In [5]:
extraction = load_snapshot("extraction", deserialize_extracted_document)
validation_raw = run_validation(extraction)
validation_payload = serialize_validated_document(validation_raw)
validation_path = write_snapshot("validation", validation_payload)
validation = load_snapshot("validation", deserialize_validated_document)

display(type(validation))
display(summarize_snapshot("validation", validation_path, validation_payload))
display(validation_payload["validations"][:3])

mellea_lrc.validation.types.ValidatedDocument

{'stage': 'validation',
 'snapshot': '../local/snapshots/bookmarked/validation.json',
 'top_level_keys': ['artifact_type',
  'citations',
  'extraction_metadata',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text',
  'validation_metadata',
  'validations']}

[{'citation_id': 'cite-0001',
  'status': 'found',
  'source': 'cl-access',
  'message': 'CourtListener: found 806 F.3d 1288 - Richard M. Villarreal v. R.J. Reynolds Tobacco Company',
  'locator': '806 F.3d 1288',
  'lookup_status': 200,
  'lookup_cache': 'hit',
  'lookup_key': '64ba4442c411106942334cadb5f409c6c79caf31e4a3bcc7c1aa87124769bdd3',
  'matches': [{'case_name': 'Richard M. Villarreal v. R.J. Reynolds Tobacco Company',
    'date_filed': '2015-11-30',
    'court': None,
    'court_id': None,
    'docket_id': '3019107',
    'extra_data': {'resource_uri': 'https://www.courtlistener.com/api/rest/v4/clusters/3160875/',
     'id': 3160875,
     'absolute_url': '/opinion/3160875/richard-m-villarreal-v-rj-reynolds-tobacco-company/',
     'panel': [],
     'non_participating_judges': [],
     'docket': 'https://www.courtlistener.com/api/rest/v4/dockets/3019107/',
     'sub_opinions': ['https://www.courtlistener.com/api/rest/v4/opinions/3160875/',
      'https://www.courtlistener.com/a

## Step 4: Assess

`run_assessment_async(ValidatedDocument)` advances the object to `AssessedDocument` and may call the configured LLM API in parallel. The snapshot is reloaded with `deserialize_assessed_document`.

In [6]:
import time

validation = load_snapshot("validation", deserialize_validated_document)
llm_config = llm_api_config_from_env(os.environ)
mellea_started: dict[str, float] = {}


def on_mellea_call(ctx: MelleaCallContext) -> None:
    mellea_started[ctx.citation_id] = time.perf_counter()
    print(
        f"[mellea] start id={ctx.citation_id} "
        f"extracted={ctx.extracted_case_name!r}",
        flush=True,
    )


def on_mellea_done(ctx: MelleaCallContext, item: CitationAssessmentResult) -> None:
    elapsed = time.perf_counter() - mellea_started[ctx.citation_id]
    case = item.case_name.initial.status.value
    followup = item.case_name.followup.status.value
    print(
        f"[mellea] done  id={ctx.citation_id} "
        f"case={case} followup={followup} ({elapsed:.1f}s)",
        flush=True,
    )


assessment_raw = await run_assessment_async(
    validation,
    mellea_concurrency=MELLEA_CONCURRENCY,
    on_mellea_call=on_mellea_call,
    on_mellea_done=on_mellea_done,
)
assessment_payload = serialize_assessed_document(assessment_raw)
assessment_path = write_snapshot("assessment", assessment_payload)
assessment = load_snapshot("assessment", deserialize_assessed_document)
assessment_status_counts = {
    status: sum(
        item.status.value == status for item in assessment.assessments
    )
    for status in sorted({item.status.value for item in assessment.assessments})
}

display(
    {
        "assessment_path": str(assessment_path),
        "assessment_records": len(assessment.assessments),
        "assessment_status_counts": assessment_status_counts,
        "all_records_terminal": assessment.assessment_complete,
    }
)

display(type(assessment))
display(
    {
        **summarize_snapshot("assessment", assessment_path, assessment_payload),
        "api_base": llm_config.api_base,
        "model": llm_config.model,
        "response_format": llm_config.response_format.value,
        "certificate_verification": llm_config.cert_required,
        "assessment_records": len(assessment.assessments),
        "assessment_status_counts": assessment_status_counts,
        "all_records_terminal": assessment.assessment_complete,
    }
)

[mellea] start id=cite-0001 extracted='United States v. Hoffman'
=== 11:42:34-WARNING ======
Mellea assumes you are NOT using the OpenAI platform, and that other model providers have less strict requirements on support JSON schemas passed into `format=`. If you encounter a server-side error following this message, then you found an exception to this assumption. Please open an issue at github.com/generative_computing/mellea with this stack trace and your inference engine / model provider.
[mellea] start id=cite-0003 extracted=None
=== 11:42:35-WARNING ======
Not using a SimpleContext with asynchronous requests could cause unexpected results due to stale contexts. Ensure you await between requests.
See the async section of the tutorial: https://github.com/generative-computing/mellea/blob/main/docs/tutorial.md#chapter-12-asynchronicity


  0%|          | 0/3 [00:00<?, ?it/s]

=== 11:42:35-WARNING ======
Mellea assumes you are NOT using the OpenAI platform, and that other model providers have less strict requirements on support JSON schemas passed into `format=`. If you encounter a server-side error following this message, then you found an exception to this assumption. Please open an issue at github.com/generative_computing/mellea with this stack trace and your inference engine / model provider.
=== 11:42:41-INFO ======
SUCCESS


  0%|          | 0/3 [00:06<?, ?it/s]

=== 11:42:41-WARNING ======
Mellea assumes you are NOT using the OpenAI platform, and that other model providers have less strict requirements on support JSON schemas passed into `format=`. If you encounter a server-side error following this message, then you found an exception to this assumption. Please open an issue at github.com/generative_computing/mellea with this stack trace and your inference engine / model provider.


...

=== 11:43:05-WARNING ======
Mellea assumes you are NOT using the OpenAI platform, and that other model providers have less strict requirements on support JSON schemas passed into `format=`. If you encounter a server-side error following this message, then you found an exception to this assumption. Please open an issue at github.com/generative_computing/mellea with this stack trace and your inference engine / model provider.


=== 11:43:06-WARNING ======
Mellea assumes you are NOT using the OpenAI platform, and that other model providers have less strict requirements on support JSON schemas passed into `format=`. If you encounter a server-side error following this message, then you found an exception to this assumption. Please open an issue at github.com/generative_computing/mellea with this stack trace and your inference engine / model provider.
[mellea] done  id=cite-0001 case=not_semantic_match followup=reassessed (32.3s)


{'assessment_path': '../local/snapshots/bookmarked/assessment.json',
 'assessment_records': 3,
 'assessment_status_counts': {'assessed': 2, 'skipped': 1},
 'all_records_terminal': True}

mellea_lrc.assessment.types.document.AssessedDocument

{'stage': 'assessment',
 'snapshot': '../local/snapshots/bookmarked/assessment.json',
 'top_level_keys': ['artifact_type',
  'assessment_metadata',
  'assessments',
  'citations',
  'extraction_metadata',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text',
  'validation_metadata',
  'validations'],
 'api_base': 'https://litellm-dev.pace.gatech.edu:4000/v1',
 'model': 'gemma-4-26B',
 'response_format': 'json_schema',
 'certificate_verification': False,
 'assessment_records': 3,
 'assessment_status_counts': {'assessed': 2, 'skipped': 1},
 'all_records_terminal': True}